# 00D — Check AWS access, one command at a time

Run the cells in order, or rerun one check after permissions change. Each check shows
its **actual boto3 call**, followed by the response or exact error. A denied check
does not stop the other checks. Only calls needing a missing prerequisite are skipped.

Only **boto3** is required. Docker, CDK and the SageMaker SDK are not needed.
The first section reads existing resources. S3 writes and the managed pipeline probe
are off until you enable their switches. Starting the pipeline runs **one small,
billable Processing job**, using synthetic values and the image you choose.

**CONFIRMED** means that specific API request succeeded. It does not establish other
permissions or mean a described job succeeded. **DENIED** shows an AWS access error;
**FAILED** includes connectivity, configuration and missing-resource errors.
**NOT TESTED** means the call was not sent.

Keep results and configured copies in your environment. Clear outputs and remove
account-specific settings before sharing this notebook.

## Configure the resources to check

- Use the current credentials and region; enter resource names supplied by your platform team.
- Set an input bucket and key even if listing buckets is denied; those checks run independently.
- Keep the pipeline orchestration role separate from the job execution role.
- Keep the write switches off for the initial read-only pass.

`input_key` is the full object key inside `input_bucket`, not an S3 URI and not relative
to `input_prefix`. `scratch_s3_uri` must name an approved bucket **and prefix**.
No bucket, repository, IAM role or infrastructure is created by this notebook.

The caller needs `iam:PassRole` for `pipeline_role_arn` when creating the pipeline.
That pipeline role needs job-creation permissions and `iam:PassRole` for `execution_role_arn`.
The job role needs access to its image, logs and configured S3 output.
[AWS pipeline role documentation](https://docs.aws.amazon.com/sagemaker/latest/dg/build-and-manage-access.html)

In [ ]:
import copy
import json
import time
import uuid
from pathlib import Path
from urllib.parse import urlparse

import boto3
from botocore.config import Config

region = ""                         # Empty uses the current AWS configuration.
input_bucket = ""
input_prefix = ""                   # Limit the object listing within that bucket.
input_key = ""                      # Exact existing object to read.
scratch_s3_uri = ""                  # e.g. s3://approved-bucket/team/access-checks/
output_kms_key_id = ""               # Empty uses the bucket's default encryption.

pipeline_role_arn = ""               # Approved pipeline orchestration role.
execution_role_arn = ""              # Approved Processing job runtime role.
existing_pipeline_name = ""         # Read-only inspection; never modified here.
existing_processing_job_name = ""    # Read-only inspection if the probe cannot run.

ecr_repository_name = ""
ecr_image_tag = "latest"
ecr_registry_id = ""                 # Empty means the current account.
image_uri = ""                      # Empty uses an AWS scikit-learn image, where known.
instance_type = "ml.m5.large"
subnet_ids = []
security_group_ids = []
volume_kms_key_id = ""
enable_network_isolation = True
required_tags = {}                  # Additional platform-required tags.

log_group_name = "/aws/sagemaker/ProcessingJobs"
existing_log_stream_prefix = ""      # Read logs even if this probe cannot create a job.
bootstrap_stack_name = "CDKToolkit"
bootstrap_version_parameter = "/cdk-bootstrap/hnb659fds/version"
wait_seconds = 900                   # Observation limit; does not stop a running job.

session = boto3.Session(region_name=region or None)
if not session.region_name:
    raise ValueError("Set region above, then rerun this setup cell.")
client_config = Config(connect_timeout=5, read_timeout=20, retries={"max_attempts": 0})
sts = session.client("sts", config=client_config)
iam = session.client("iam", config=client_config)
s3 = session.client("s3", config=client_config)
sm = session.client("sagemaker", config=client_config)
ecr = session.client("ecr", config=client_config)
logs = session.client("logs", config=client_config)
cfn = session.client("cloudformation", config=client_config)
ssm = session.client("ssm", config=client_config)

probe_id = uuid.uuid4().hex
pipeline_name = "managed-access-" + probe_id[:10]
report_dir = Path.cwd() / ".runs" / ("access-" + probe_id[:10])
results = {}
identity = None
pipeline_arn = None
execution_arn = None
processing_job_name = existing_processing_job_name
processing_status = None
expected_revision = "1"
probe_written = False
probe_version_id = None
cleanup_tags_verified = False
cleanup_executions_clear = False
print("Region:", session.region_name)
print("Reports:", report_dir.resolve())
print("Clients are ready; the access checks are below. Do not rerun setup while a probe is active.")

## Print and save each result

- Define one display helper; it does not call AWS or decide which commands to run.
- Print the complete error, including the denied action and resource when AWS supplies them.
- Save observed results after each check so a later failure does not lose earlier evidence.

In [ ]:
def show_result(action, response=None, *, error=None, skipped=None):
    """Take an API result or error, print it, and save it locally."""
    if error is not None:
        payload = getattr(error, "response", {})
        error_code = payload.get("Error", {}).get("Code", type(error).__name__)
        denied = error_code in {"AccessDenied", "AccessDeniedException", "UnauthorizedOperation", "403"}
        result = {"status": "DENIED" if denied else "FAILED",
                  "error_type": type(error).__name__, "error_code": error_code,
                  "message": str(error), "aws_error": payload}
    elif skipped is not None:
        result = {"status": "NOT TESTED", "reason": skipped}
    else:
        result = {"status": "CONFIRMED", "response": response}
    results[action] = result
    print(action, "—", result["status"])
    print(json.dumps(result, indent=2, default=str))
    report_dir.mkdir(parents=True, exist_ok=True)
    (report_dir / "results.json").write_text(json.dumps(results, indent=2, default=str) + "\n")

print("Display helper ready. The AWS commands are in the following cells.")

## Who is calling AWS?

- Call STS using the notebook's current credentials.
- Show the account and caller ARN without guessing a job execution role.
- Continue other API checks even if this identity lookup fails.

In [ ]:
action = 'sts:GetCallerIdentity'
try:
    response = sts.get_caller_identity()
    identity = response
    show_result(action, response)
except Exception as error:
    show_result(action, error=error)

## Read the current IAM role

- Extract the role name from a successful STS assumed-role response.
- Call IAM GetRole to obtain the actual ARN, including any IAM path.
- Treat denied IAM access or an unreachable IAM endpoint separately from SageMaker access.

In [ ]:
action = 'iam:GetRole'
if identity and ':assumed-role/' in identity['Arn']:
    caller_role_name = identity['Arn'].split(':assumed-role/', 1)[1].split('/')[0]
    print('RoleName:', caller_role_name)
    try:
        response = iam.get_role(RoleName=caller_role_name)
        show_result(action, response)
    except Exception as error:
        show_result(action, error=error)
else:
    show_result(action, skipped='No STS assumed-role identity is available; other checks still run.')

## List buckets

- Send ListBuckets and show up to ten buckets owned by the current account.
- This tests s3:ListAllMyBuckets, not read access to a particular bucket.
- A denial here does not skip the named-bucket and object checks below.

In [ ]:
action = 's3:ListAllMyBuckets / ListBuckets'
try:
    response = s3.list_buckets(MaxBuckets=10)
    show_result(action, response)
except Exception as error:
    show_result(action, error=error)

## Read the selected bucket's region

- Call GetBucketLocation for the configured input bucket.
- Show its region; a null LocationConstraint denotes us-east-1.
- Continue the object checks if this permission is denied.

In [ ]:
action = 's3:GetBucketLocation'
if input_bucket:
    print('Bucket:', input_bucket)
    try:
        response = s3.get_bucket_location(Bucket=input_bucket)
        show_result(action, response)
    except Exception as error:
        show_result(action, error=error)
else:
    show_result(action, skipped='Set input_bucket.')

## List objects in the selected prefix

- Call ListObjectsV2 for the configured bucket and prefix.
- Show at most five object records; this is a permission probe, not a complete inventory.
- Continue reading the exact configured object even if listing is denied.

In [ ]:
action = 's3:ListBucket / ListObjectsV2'
if input_bucket:
    print('Bucket:', input_bucket, 'Prefix:', repr(input_prefix))
    try:
        response = s3.list_objects_v2(Bucket=input_bucket, Prefix=input_prefix, MaxKeys=5)
        show_result(action, response)
    except Exception as error:
        show_result(action, error=error)
else:
    show_result(action, skipped='Set input_bucket.')

## Inspect the selected object

- Call HeadObject for the exact configured key.
- Show metadata such as size, encryption and version without reading its data.
- The next cell tests an actual data read independently of this response.

In [ ]:
action = 's3:GetObject / HeadObject'
if input_bucket and input_key:
    print('Object:', f's3://{input_bucket}/{input_key}')
    try:
        response = s3.head_object(Bucket=input_bucket, Key=input_key)
        show_result(action, response)
    except Exception as error:
        show_result(action, error=error)
else:
    show_result(action, skipped='Set input_bucket and input_key.')

## Read the selected object

- Call GetObject for the configured key even if bucket listing or HeadObject failed.
- Read up to 1 KiB, then close the response stream.
- Print only metadata and the byte count; do not display the input data.

In [ ]:
action = 's3:GetObject / input'
if input_bucket and input_key:
    print('Object:', f's3://{input_bucket}/{input_key}')
    try:
        response = s3.get_object(Bucket=input_bucket, Key=input_key)
        with response["Body"] as body:
            bytes_read = len(body.read(1024))
        show_result(action, {"bytes_read": bytes_read, "size": response.get("ContentLength"),
                             "version_id": response.get("VersionId"), "etag": response.get("ETag"),
                             "request_id": response.get("ResponseMetadata", {}).get("RequestId")})
    except Exception as error:
        show_result(action, error=error)
else:
    show_result(action, skipped='Set input_bucket and input_key.')

## List SageMaker pipelines

- Call ListPipelines without trying to create anything.
- Show up to five pipeline summaries.
- This establishes listing access only, not CreatePipeline, UpdatePipeline or StartPipelineExecution.

In [ ]:
action = 'sagemaker:ListPipelines'
try:
    response = sm.list_pipelines(MaxResults=5)
    show_result(action, response)
except Exception as error:
    show_result(action, error=error)

## Describe an existing pipeline

- Read the existing pipeline named in the configuration.
- Inspect its role and definition without changing or executing it.
- This check does not depend on creating the notebook's probe pipeline.

In [ ]:
action = 'sagemaker:DescribePipeline / existing'
if existing_pipeline_name:
    print('PipelineName:', existing_pipeline_name)
    try:
        response = sm.describe_pipeline(PipelineName=existing_pipeline_name)
        show_result(action, response)
    except Exception as error:
        show_result(action, error=error)
else:
    show_result(action, skipped='Set existing_pipeline_name.')

## List Processing jobs

- Call ListProcessingJobs independently of pipeline creation.
- Show up to five existing records.
- Listing success does not prove permission to create jobs or register models.

In [ ]:
action = 'sagemaker:ListProcessingJobs'
try:
    response = sm.list_processing_jobs(MaxResults=5)
    show_result(action, response)
except Exception as error:
    show_result(action, error=error)

## List Training jobs

- Call ListTrainingJobs independently of pipeline creation.
- Show up to five existing records.
- Listing success does not prove permission to create jobs or register models.

In [ ]:
action = 'sagemaker:ListTrainingJobs'
try:
    response = sm.list_training_jobs(MaxResults=5)
    show_result(action, response)
except Exception as error:
    show_result(action, error=error)

## List Batch Transform jobs

- Call ListTransformJobs independently of pipeline creation.
- Show up to five existing records.
- Listing success does not prove permission to create jobs or register models.

In [ ]:
action = 'sagemaker:ListTransformJobs'
try:
    response = sm.list_transform_jobs(MaxResults=5)
    show_result(action, response)
except Exception as error:
    show_result(action, error=error)

## List model registry groups

- Call ListModelPackageGroups independently of pipeline creation.
- Show up to five existing records.
- Listing success does not prove permission to create jobs or register models.

In [ ]:
action = 'sagemaker:ListModelPackageGroups'
try:
    response = sm.list_model_package_groups(MaxResults=5)
    show_result(action, response)
except Exception as error:
    show_result(action, error=error)

## Inspect an ECR repository

- Call DescribeRepositories for a repository you select.
- Use ecr_registry_id only if the repository belongs to another account.
- This checks repository metadata access; it does not test image push or layer downloads.

In [ ]:
action = 'ecr:DescribeRepositories'
if ecr_repository_name:
    registry_argument = {'registryId': ecr_registry_id} if ecr_registry_id else {}
    print('Repository:', ecr_repository_name, 'Registry:', ecr_registry_id or 'current account')
    try:
        response = ecr.describe_repositories(repositoryNames=[ecr_repository_name], **registry_argument)
        show_result(action, response)
    except Exception as error:
        show_result(action, error=error)
else:
    show_result(action, skipped='Set ecr_repository_name.')

## Inspect an ECR image

- Call DescribeImages for one configured image tag.
- Run even if DescribeRepositories failed.
- Show its digest and metadata without retrieving a login token.

In [ ]:
action = 'ecr:DescribeImages'
if ecr_repository_name and ecr_image_tag:
    registry_argument = {'registryId': ecr_registry_id} if ecr_registry_id else {}
    print('Repository:', ecr_repository_name, 'Tag:', ecr_image_tag)
    try:
        response = ecr.describe_images(repositoryName=ecr_repository_name,
            imageIds=[{'imageTag': ecr_image_tag}], **registry_argument)
        show_result(action, response)
    except Exception as error:
        show_result(action, error=error)
else:
    show_result(action, skipped='Set ecr_repository_name and ecr_image_tag.')

## List CloudWatch log streams

- Call DescribeLogStreams for the selected log group.
- Show up to five stream records without reading their messages.
- A missing log group is reported as a failed lookup, not as a permission denial.

In [ ]:
action = 'logs:DescribeLogStreams'
if log_group_name:
    print('LogGroupName:', log_group_name)
    try:
        response = logs.describe_log_streams(logGroupName=log_group_name, limit=5)
        show_result(action, response)
    except Exception as error:
        show_result(action, error=error)
else:
    show_result(action, skipped='Set log_group_name.')

## Read the CDK bootstrap stack

- Call CloudFormation DescribeStacks for the configured bootstrap stack.
- This check is independent of all SageMaker and S3 checks.
- Success establishes read access only; it does not establish CDK deployment permission.

In [ ]:
action = 'cloudformation:DescribeStacks'
if bootstrap_stack_name:
    print('StackName:', bootstrap_stack_name)
    try:
        response = cfn.describe_stacks(StackName=bootstrap_stack_name)
        show_result(action, response)
    except Exception as error:
        show_result(action, error=error)
else:
    show_result(action, skipped='Set bootstrap_stack_name.')

## Read the CDK bootstrap version

- Call SSM GetParameter for the configured CDK bootstrap version.
- Request no secret decryption.
- A missing parameter, denied access and connectivity failure remain distinct errors.

In [ ]:
action = 'ssm:GetParameter'
if bootstrap_version_parameter:
    print('Parameter:', bootstrap_version_parameter)
    try:
        response = ssm.get_parameter(Name=bootstrap_version_parameter, WithDecryption=False)
        show_result(action, response)
    except Exception as error:
        show_result(action, error=error)
else:
    show_result(action, skipped='Set bootstrap_version_parameter.')

## Enable the S3 write probe

- Leave `WRITE_S3_PROBE` false to skip writing; all read-only checks still run.
- Use only the configured scratch prefix and a new key containing this notebook's random ID.
- Save that object version so the separate cleanup cell can remove exactly the version written.

In [ ]:
WRITE_S3_PROBE = False

scratch = urlparse(scratch_s3_uri)
scratch_configured = bool(
    scratch.scheme == "s3" and scratch.netloc and scratch.path.strip("/")
    and not scratch.query and not scratch.fragment
)
scratch_bucket = scratch.netloc
scratch_prefix = scratch.path.lstrip("/").rstrip("/")
probe_key = scratch_prefix + "/" + pipeline_name + "/s3-probe.txt"
probe_bytes = ("Access check " + probe_id + "\n").encode()
output_s3_uri = (
    f"s3://{scratch_bucket}/{scratch_prefix}/{pipeline_name}/processing/"
    if scratch_configured else ""
)
print("S3 writes enabled:", WRITE_S3_PROBE)
print("Scratch prefix:", scratch_s3_uri or "not configured")
if scratch_s3_uri and not scratch_configured:
    print("Invalid scratch_s3_uri: supply s3://bucket/nonempty-prefix/.")

## Write one scratch object

- Call PutObject only after enabling writes and configuring a scratch prefix.
- Use IfNoneMatch to avoid overwriting an object at that key.
- Use the configured KMS key if supplied; otherwise keep bucket-default encryption.

In [ ]:
action = 's3:PutObject'
if WRITE_S3_PROBE and scratch_configured and not probe_written:
    print('Object:', f's3://{scratch_bucket}/{probe_key}')
    encryption_arguments = ({'ServerSideEncryption': 'aws:kms', 'SSEKMSKeyId': output_kms_key_id}
                            if output_kms_key_id else {})
    try:
        response = s3.put_object(
            Bucket=scratch_bucket, Key=probe_key, Body=probe_bytes,
            ContentType="text/plain", IfNoneMatch="*", **encryption_arguments
        )
        probe_written = True
        probe_version_id = response.get('VersionId')
        show_result(action, response)
    except Exception as error:
        show_result(action, error=error)
else:
    show_result(action, skipped='Enable WRITE_S3_PROBE and configure scratch_s3_uri; an already confirmed write is not repeated.')

## Read back the scratch object

- Call GetObject for the object this notebook successfully wrote.
- Pin the version when S3 returned a version ID.
- Compare the returned bytes with the small synthetic message.

In [ ]:
action = 's3:GetObject / scratch'
if probe_written:
    version_argument = {'VersionId': probe_version_id} if probe_version_id else {}
    print('Object:', f's3://{scratch_bucket}/{probe_key}', 'Version:', probe_version_id)
    try:
        response = s3.get_object(Bucket=scratch_bucket, Key=probe_key, **version_argument)
        with response["Body"] as body:
            returned_bytes = body.read(len(probe_bytes) + 1)
        if returned_bytes != probe_bytes:
            raise ValueError("GetObject succeeded, but the returned bytes differ from this probe.")
        show_result(action, {"bytes_match": True, "version_id": response.get("VersionId")})
    except Exception as error:
        show_result(action, error=error)
else:
    show_result(action, skipped='No scratch write was confirmed.')

## Prepare the one-step managed pipeline

- Leave `CREATE_PROBE_PIPELINE` and `START_PROBE_EXECUTION` false for a read-only pass.
- Inspect the literal Processing step below; it sums two synthetic values and writes `result.json`.
- Set distinct approved pipeline and job roles, plus an approved image and any required VPC/KMS settings.
- Use scratch S3 output when configured; otherwise verify the result through CloudWatch.

The default image map comes from the
[SageMaker SDK's scikit-learn image configuration](https://github.com/aws/sagemaker-python-sdk/blob/v2.253.1/src/sagemaker/image_uri_config/sklearn.json).
These registry IDs belong to AWS. Set `image_uri` if your platform requires a different
approved Python image. `CREATE_PROBE_PIPELINE` also enables updating **this probe only**.
No existing pipeline named earlier in the notebook is modified.

In [ ]:
CREATE_PROBE_PIPELINE = False
START_PROBE_EXECUTION = False

AWS_IMAGE_REGISTRIES = {
    "af-south-1": "510948584623",
    "ap-east-1": "651117190479",
    "ap-northeast-1": "354813040037",
    "ap-northeast-2": "366743142698",
    "ap-northeast-3": "867004704886",
    "ap-south-1": "720646828776",
    "ap-south-2": "628508329040",
    "ap-southeast-1": "121021644041",
    "ap-southeast-2": "783357654285",
    "ap-southeast-3": "951798379941",
    "ap-southeast-4": "106583098589",
    "ca-central-1": "341280168497",
    "ca-west-1": "190319476487",
    "eu-central-1": "492215442770",
    "eu-central-2": "680994064768",
    "eu-north-1": "662702820516",
    "eu-south-1": "978288397137",
    "eu-south-2": "104374241257",
    "eu-west-1": "141502667606",
    "eu-west-2": "764974769150",
    "eu-west-3": "659782779980",
    "il-central-1": "898809789911",
    "me-central-1": "272398656194",
    "me-south-1": "801668240914",
    "sa-east-1": "737474898029",
    "us-east-1": "683313688378",
    "us-east-2": "257758044811",
    "us-west-1": "746614075791",
    "us-west-2": "246618743249"
}
selected_registry = AWS_IMAGE_REGISTRIES.get(session.region_name)
selected_image = image_uri or (
    f"{selected_registry}.dkr.ecr.{session.region_name}.amazonaws.com/sagemaker-scikit-learn:1.2-1-cpu-py3"
    if selected_registry else ""
)
script = '''
import json, os
from pathlib import Path
print("PROBE_ENTRY", flush=True)
result = {"probe": os.environ["PROBE_ID"], "revision": os.environ["PROBE_REVISION"],
          "rows": 2, "sum": sum([10, 20])}
output = Path("/opt/ml/processing/output")
output.mkdir(parents=True, exist_ok=True)
(output / "result.json").write_text(json.dumps(result))
print("PROBE_RESULT " + json.dumps(result), flush=True)
print("PROBE_SUCCESS", flush=True)
'''
# Remove the indentation from this readable embedded script.
import textwrap
script = textwrap.dedent(script)
cluster = {"InstanceCount": 1, "InstanceType": instance_type, "VolumeSizeInGB": 10}
if volume_kms_key_id:
    cluster["VolumeKmsKeyId"] = volume_kms_key_id
job_arguments = {
    "ProcessingJobName": pipeline_name + "-job",
    "RoleArn": execution_role_arn,
    "AppSpecification": {
        "ImageUri": selected_image,
        "ContainerEntrypoint": ["python3", "-c", "import sys;exec(''.join(sys.argv[1:]))"],
        # SageMaker limits each entrypoint/argument string to 256 characters.
        "ContainerArguments": [script[i:i + 256] for i in range(0, len(script), 256)],
    },
    "Environment": {"PROBE_ID": probe_id, "PROBE_REVISION": {"Get": "Parameters.ProbeRevision"}},
    "ProcessingResources": {"ClusterConfig": cluster},
    "StoppingCondition": {"MaxRuntimeInSeconds": 120},
    "NetworkConfig": {"EnableNetworkIsolation": enable_network_isolation},
}
if subnet_ids and security_group_ids:
    job_arguments["NetworkConfig"]["VpcConfig"] = {
        "Subnets": subnet_ids, "SecurityGroupIds": security_group_ids,
    }
if output_s3_uri:
    job_arguments["ProcessingOutputConfig"] = {"Outputs": [{
        "OutputName": "result",
        "S3Output": {"S3Uri": output_s3_uri, "LocalPath": "/opt/ml/processing/output",
                     "S3UploadMode": "EndOfJob"},
    }]}
    if output_kms_key_id:
        job_arguments["ProcessingOutputConfig"]["KmsKeyId"] = output_kms_key_id
definition = {
    "Version": "2020-12-01",
    "Parameters": [{"Name": "ProbeRevision", "Type": "String", "DefaultValue": "1"}],
    "Steps": [{"Name": "CheckAccess", "Type": "Processing", "Arguments": job_arguments}],
}
probe_settings_valid = bool(
    pipeline_role_arn and execution_role_arn and pipeline_role_arn != execution_role_arn
    and selected_image and bool(subnet_ids) == bool(security_group_ids)
    and (not scratch_s3_uri or scratch_configured)
)
tags = {**required_tags, "Purpose": "ManagedAccessProbe", "ProbeId": probe_id}
print("Probe pipeline:", pipeline_name)
print("Pipeline role:", pipeline_role_arn or "not configured")
print("Job role:", execution_role_arn or "not configured")
print("Image:", selected_image or "not configured")
print("Settings ready:", probe_settings_valid)
print(json.dumps(definition, indent=2))

## Create the probe pipeline

- Call CreatePipeline with the separate pipeline orchestration role.
- Show the exact iam:PassRole error if the caller cannot pass that role.
- Skip only the probe-dependent calls if creation fails; independent checks remain usable.

In [ ]:
action = 'sagemaker:CreatePipeline'
if CREATE_PROBE_PIPELINE and probe_settings_valid and not pipeline_arn:
    print('PipelineName:', pipeline_name, '\nRoleArn:', pipeline_role_arn)
    try:
        response = sm.create_pipeline(
            PipelineName=pipeline_name, RoleArn=pipeline_role_arn,
            PipelineDefinition=json.dumps(definition),
            PipelineDescription="Synthetic managed access check",
            Tags=[{"Key": key, "Value": value} for key, value in tags.items()],
            ClientRequestToken=probe_id,
        )
        pipeline_arn = response['PipelineArn']
        show_result(action, response)
    except Exception as error:
        show_result(action, error=error)
else:
    show_result(action, skipped='Enable CREATE_PROBE_PIPELINE and set distinct approved roles/image plus valid VPC/S3 settings; do not recreate a confirmed probe.')

## Update the probe pipeline

- Call UpdatePipeline only for the pipeline created in this kernel.
- Change the ProbeRevision default from 1 to 2 so the container result can confirm the update.
- If updating is denied, StartPipelineExecution below can still test the original pipeline.

In [ ]:
action = 'sagemaker:UpdatePipeline'
if CREATE_PROBE_PIPELINE and pipeline_arn:
    updated_definition = copy.deepcopy(definition)
    updated_definition['Parameters'][0]['DefaultValue'] = '2'
    print('PipelineName:', pipeline_name, '\nRoleArn:', pipeline_role_arn)
    try:
        response = sm.update_pipeline(
            PipelineName=pipeline_name, RoleArn=pipeline_role_arn,
            PipelineDefinition=json.dumps(updated_definition),
            PipelineDescription="Synthetic managed access check, revision 2",
        )
        expected_revision = '2'
        show_result(action, response)
    except Exception as error:
        show_result(action, error=error)
else:
    show_result(action, skipped='No probe pipeline creation is confirmed, or CREATE_PROBE_PIPELINE is off.')

## Describe the probe pipeline

- Call DescribePipeline for the newly created probe.
- Show the stored definition and orchestration role.
- Run even if UpdatePipeline failed.

In [ ]:
action = 'sagemaker:DescribePipeline / probe'
if pipeline_arn:
    try:
        response = sm.describe_pipeline(PipelineName=pipeline_name)
        show_result(action, response)
    except Exception as error:
        show_result(action, error=error)
else:
    show_result(action, skipped='No probe pipeline creation is confirmed.')

## Start the probe execution

- Enable START_PROBE_EXECUTION above to start one billable Processing job.
- Call StartPipelineExecution even if the update or describe checks failed.
- Reuse this kernel's request token on a retry to avoid starting another execution.

In [ ]:
action = 'sagemaker:StartPipelineExecution'
if START_PROBE_EXECUTION and pipeline_arn:
    print('PipelineName:', pipeline_name)
    try:
        response = sm.start_pipeline_execution(
            PipelineName=pipeline_name,
            PipelineExecutionDisplayName="access-" + probe_id[:10],
            ClientRequestToken=probe_id,
        )
        execution_arn = response['PipelineExecutionArn']
        show_result(action, response)
    except Exception as error:
        show_result(action, error=error)
else:
    show_result(action, skipped='Enable START_PROBE_EXECUTION and successfully create the probe first.')

## Describe the execution until it finishes

- Call DescribePipelineExecution repeatedly, showing each status change.
- Print FailureReason if the pipeline fails; a successful describe request is not a successful job.
- Stop observing after wait_seconds without stopping the actual execution; rerun this cell to keep watching.

In [ ]:
action = "sagemaker:DescribePipelineExecution"
if execution_arn:
    deadline = time.monotonic() + wait_seconds
    previous_status = None
    try:
        while True:
            response = sm.describe_pipeline_execution(PipelineExecutionArn=execution_arn)
            status = response["PipelineExecutionStatus"]
            if status != previous_status:
                print("Execution:", execution_arn, "Status:", status)
                previous_status = status
            if status not in {"Executing", "Stopping"}:
                break
            if time.monotonic() >= deadline:
                print("Observation ended. Execution is still active; rerun this cell or inspect Studio.")
                break
            time.sleep(10)
        show_result(action, response)
    except Exception as error:
        show_result(action, error=error)
else:
    show_result(action, skipped="No probe execution start is confirmed.")

## Inspect the execution's steps

- Call ListPipelineExecutionSteps even if polling failed.
- Show any step FailureReason, including job-role, image or S3 access errors.
- Capture the Processing job name when the service created one.

In [ ]:
action = 'sagemaker:ListPipelineExecutionSteps'
if execution_arn:
    try:
        response = sm.list_pipeline_execution_steps(PipelineExecutionArn=execution_arn)
        for step in response.get("PipelineExecutionSteps", []):
            job_arn = step.get("Metadata", {}).get("ProcessingJob", {}).get("Arn")
            if job_arn:
                processing_job_name = job_arn.rsplit("/", 1)[-1]
        show_result(action, response)
    except Exception as error:
        show_result(action, error=error)
else:
    show_result(action, skipped='No probe execution start is confirmed.')

## Describe a Processing job

- Use this probe's job name, or existing_processing_job_name supplied in setup.
- Call DescribeProcessingJob and show the runtime role, job status and any FailureReason.
- An existing job can be inspected even when CreatePipeline failed.

In [ ]:
action = 'sagemaker:DescribeProcessingJob'
if processing_job_name:
    print('ProcessingJobName:', processing_job_name)
    try:
        response = sm.describe_processing_job(ProcessingJobName=processing_job_name)
        processing_status = response.get('ProcessingJobStatus')
        show_result(action, response)
    except Exception as error:
        show_result(action, error=error)
else:
    show_result(action, skipped='No probe job name is available; set existing_processing_job_name to inspect another job.')

## Read CloudWatch messages

- Read this probe job's messages, or the configured existing_log_stream_prefix.
- Call FilterLogEvents directly, even if listing streams or describing the job failed.
- Show at most 100 messages; logs can arrive late, so rerun if needed and inspect any nextToken.

In [ ]:
action = 'logs:FilterLogEvents'
if log_group_name and (processing_job_name or existing_log_stream_prefix):
    stream_prefix = processing_job_name + '/' if processing_job_name else existing_log_stream_prefix
    print('LogGroupName:', log_group_name, 'LogStreamNamePrefix:', stream_prefix)
    try:
        response = logs.filter_log_events(logGroupName=log_group_name, logStreamNamePrefix=stream_prefix, limit=100)
        show_result(action, response)
    except Exception as error:
        show_result(action, error=error)
else:
    show_result(action, skipped='No job name or existing_log_stream_prefix is available.')

## Read the managed job's S3 result

- Call GetObject for this probe's result.json if an execution was started with S3 output.
- Check its probe ID, executed revision, row count and sum.
- A missing object while the job is still running is not a permission denial; rerun after completion.

In [ ]:
action = 's3:GetObject / managed output'
if execution_arn and output_s3_uri:
    output_location = urlparse(output_s3_uri + 'result.json')
    print('Object:', output_location.geturl())
    try:
        response = s3.get_object(Bucket=output_location.netloc, Key=output_location.path.lstrip('/'))
        with response["Body"] as body:
            result = json.loads(body.read(4096))
        expected = {"probe": probe_id, "revision": expected_revision, "rows": 2, "sum": 30}
        if result != expected:
            raise ValueError(f"GetObject succeeded, but result was {result!r}; expected {expected!r}.")
        show_result(action, result)
    except Exception as error:
        show_result(action, error=error)
else:
    show_result(action, skipped='No execution start is confirmed, or scratch_s3_uri was not configured.')

## Optional cleanup

- Keep the switches false while inspecting resources.
- Stop only the execution started by this notebook; rerun the describe/wait cell after requesting a stop.
- Delete only this kernel's tagged probe pipeline and the exact scratch object version it wrote.
- Retain managed job output, job history and logs; no existing pipeline or input object is removed.

Run the cleanup cells in order. Pipeline deletion requires a successful ownership check
and a complete execution list with no active executions. Cleanup checks have their own
errors; they do not erase the earlier access results.

In [ ]:
STOP_PROBE_EXECUTION = False
DELETE_PROBE_PIPELINE = False
DELETE_SCRATCH_OBJECT = False

cleanup_tags_verified = False
cleanup_executions_clear = False
print("Stop execution:", STOP_PROBE_EXECUTION)
print("Delete pipeline:", DELETE_PROBE_PIPELINE)
print("Delete scratch object:", DELETE_SCRATCH_OBJECT)

## Stop this notebook's execution

- Call StopPipelineExecution only when explicitly enabled and an execution ARN was recorded.
- Stopping is asynchronous; rerun the describe/wait cell until the status is terminal.
- The following cleanup checks still report their own results if stopping is denied.

In [ ]:
action = 'sagemaker:StopPipelineExecution'
if STOP_PROBE_EXECUTION and execution_arn:
    try:
        response = sm.stop_pipeline_execution(PipelineExecutionArn=execution_arn, ClientRequestToken=probe_id)
        show_result(action, response)
    except Exception as error:
        show_result(action, error=error)
else:
    show_result(action, skipped='STOP_PROBE_EXECUTION is off, or no probe execution start is confirmed.')

## Check the probe's ownership tags

- Call ListTags for the exact pipeline ARN returned by this notebook's CreatePipeline request.
- Require both the random ProbeId and the ManagedAccessProbe purpose tag.
- Do not authorize deletion if this check fails.

In [ ]:
action = 'sagemaker:ListTags / cleanup'
if DELETE_PROBE_PIPELINE and pipeline_arn:
    cleanup_tags_verified = False
    try:
        response = sm.list_tags(ResourceArn=pipeline_arn)
        actual_tags = {tag["Key"]: tag["Value"] for tag in response.get("Tags", [])}
        cleanup_tags_verified = (actual_tags.get("ProbeId") == probe_id
                                 and actual_tags.get("Purpose") == "ManagedAccessProbe")
        show_result(action, response)
        print("Ownership verified:", cleanup_tags_verified)
    except Exception as error:
        show_result(action, error=error)
else:
    show_result(action, skipped='DELETE_PROBE_PIPELINE is off, or no probe pipeline creation is confirmed.')

## Check for active probe executions

- Call ListPipelineExecutions only for this kernel's probe pipeline.
- Refuse deletion if any execution is active, or if the response has another page.
- Rerun after a requested stop has finished.

In [ ]:
action = 'sagemaker:ListPipelineExecutions / cleanup'
if DELETE_PROBE_PIPELINE and pipeline_arn:
    cleanup_executions_clear = False
    try:
        response = sm.list_pipeline_executions(PipelineName=pipeline_name, MaxResults=100)
        active = [item for item in response.get("PipelineExecutionSummaries", [])
                  if item["PipelineExecutionStatus"] in {"Executing", "Stopping"}]
        cleanup_executions_clear = not active and not response.get("NextToken")
        show_result(action, response)
        print("Complete execution list, with none active:", cleanup_executions_clear)
    except Exception as error:
        show_result(action, error=error)
else:
    show_result(action, skipped='DELETE_PROBE_PIPELINE is off, or no probe pipeline creation is confirmed.')

## Delete the probe pipeline

- Call DeletePipeline only after the ownership and execution checks pass.
- Never delete the separate existing_pipeline_name from setup.
- Leave Processing history, logs and managed output available for inspection.

In [ ]:
action = 'sagemaker:DeletePipeline'
if DELETE_PROBE_PIPELINE and pipeline_arn and cleanup_tags_verified and cleanup_executions_clear:
    print('PipelineName:', pipeline_name)
    try:
        response = sm.delete_pipeline(PipelineName=pipeline_name, ClientRequestToken=probe_id)
        pipeline_arn = None
        cleanup_tags_verified = False
        cleanup_executions_clear = False
        show_result(action, response)
    except Exception as error:
        show_result(action, error=error)
else:
    show_result(action, skipped='Enable DELETE_PROBE_PIPELINE and pass the two cleanup checks above.')

## Delete the scratch object version

- Call DeleteObject only for the synthetic object whose write was confirmed in this kernel.
- Pass its VersionId when available, removing that version rather than adding a delete marker.
- Do not delete input data or managed Processing output.

In [ ]:
action = 's3:DeleteObject / scratch'
if DELETE_SCRATCH_OBJECT and probe_written:
    version_argument = {'VersionId': probe_version_id} if probe_version_id else {}
    print('Object:', f's3://{scratch_bucket}/{probe_key}', 'Version:', probe_version_id)
    try:
        response = s3.delete_object(Bucket=scratch_bucket, Key=probe_key, **version_argument)
        probe_written = False
        show_result(action, response)
    except Exception as error:
        show_result(action, error=error)
else:
    show_result(action, skipped='DELETE_SCRATCH_OBJECT is off, or no scratch write was confirmed.')

## Review what was actually tested

- Show the latest result for each cell you ran, without hiding its detailed output above.
- Use the exact AWS errors in results.json when discussing access with your platform team.
- Keep untested permissions explicit: listing is not creation, and this probe executes Processing only.

This notebook does **not** execute Training or Batch Transform, register or approve models,
push/pull image layers, or deploy/destroy infrastructure. It does not grant permissions.
KMS and VPC settings are exercised only when the enabled checks actually use them.

In [ ]:
for action, result in results.items():
    print(f"{result['status']:11} {action}")
print("\nDetailed results:", (report_dir / "results.json").resolve())
print("NOT TESTED means no request was sent; it does not mean access was denied.")
print("Do not share outputs or configured copies outside your environment.")